# KíLín dogfooding: does it answer questions about its own code correctly?

**The Problem.** KíLín's system prompt promises to answer only from the supplied evidence,
cite claims with `[1]`, `[2]`, and say when the evidence is insufficient. Does KíLín's real,
end-to-end chat pipeline -- retrieval, prompt construction, generation, all of it -- actually honor
that contract when asked real questions about a real codebase?

**The Data.** KíLín's own `kilin/` package *and* its `tutorials/` notebooks, opened as a real
`type_id=code` project against the live repository via `prepare_project_workspace`. 22 hand-authored questions
(`benchmarks/gold/dogfood_kilin_code.json`), with source-checked citation candidates
and a fact checklist for grading, plus a reference reply written from that checklist as a comparison
aid for Tier 2 grading. Five are deliberate negatives.

**The Metrics.** Two tiers. Tier 1 is fully mechanical and runs automatically in this
notebook: citation validity (every `[n]` in an answer refers to evidence KíLín actually supplied),
citation coverage (of a question's gold-relevant evidence, how much actually got cited), abstention
correctness on the negative questions, and verbatim-copy detection (the system prompt forbids
copying source text directly). Tier 2 -- does each cited claim actually hold up, does the answer
satisfy its fact checklist -- needs judgment a regex can't provide, and this notebook exports it as a
worksheet for you to grade by hand.

**Reproducibility.** This benchmark evaluates a fixed code-explanation intent (`USE_ROUTER=False`) for evidence and citation analysis. Routing can be enabled separately, but changes what is being measured. The KíLín gold annotations are tied to the October 23, 2026 source snapshot and must be rechecked after source changes. The original QAChat gold file is retained unchanged as historical data.

## 1. Prepare KíLín

Locate the KíLín repo root, put it on the import path, and run KíLín's notebook setup.


In [ ]:
from pathlib import Path

root = Path.cwd()
while not (root / "kilin").is_dir() and root != root.parent:
    root = root.parent

if not (root / "kilin").is_dir():
    raise RuntimeError("Open the notebook from the KíLín checkout.")

# The benchmark helpers and run folders live in _si_article_/benchmarks/,
# imported as the benchmarks package from its parent folder.
import sys

benchmarks_dir = root / "_si_article_" / "benchmarks"
if str(benchmarks_dir.parent) not in sys.path:
    sys.path.insert(0, str(benchmarks_dir.parent))

print(f"KíLín root: {root}")

In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)

## 2. Load the gold question set

The questions and their gold citations are hand-authored and checked
into the repo (`benchmarks/gold/dogfood_kilin_code.json`), since only a human who actually knows this
codebase can write trustworthy ground truth for it. `load_generation_tasks` (`benchmarks/common.py`)
turns each entry into a `GenerationTask`.


In [ ]:
from benchmarks.common import load_generation_tasks

gold_path = benchmarks_dir / "gold" / "dogfood_kilin_code.json"
tasks = load_generation_tasks(gold_path)

abstention_count = sum(1 for task in tasks if task.requires_abstention)
print(
    f"Loaded {len(tasks)} questions "
    f"({abstention_count} deliberate abstention questions)."
)
for task in tasks:
    marker = " [abstention]" if task.requires_abstention else ""
    print(f"  {task.task_id}{marker}: {task.question}")

## 3. Point KíLín at its own repository -- package and tutorials

`prepare_project_workspace` is the real function KíLín uses to open an existing folder as a project.
It takes one `source_directory`, so covering both `kilin/` (the package)
and `tutorials/` (KíLín's own 13 usage-tutorial notebooks) needs a small
staging step first: the two folders are siblings at the repo root, alongside `benchmarks/`, `tests/`,
`setup/`, and -- critically -- `sandbox/` itself, so pointing `source_directory` at the repo root
directly isn't safe (KíLín would try to mirror `sandbox/` into itself) or useful (it would sweep in
this benchmark's own harness as noise).

So the cell below copies just `kilin/` and `tutorials/` into a combined staging folder under
`benchmarks/sandbox/_staging/` first, and that's what gets handed to `prepare_project_workspace` as the
real source -- mirrored into a managed workspace tracked with a SHA-256 manifest, same as before, just
with one extra copy pass ahead of it.

**This is a new source path, so it gets a fresh `project_kilin_and_tutorials/` workspace** .
Each run allocates a new workspace with default profile settings, which
would silently drop this session's chat/embedding model tuning back to KíLín's stock defaults -- exactly
the stale-config failure mode this notebook's runs kept hitting earlier. The cell below enforces the
known-good settings explicitly right after opening the project.

In [ ]:
import shutil

from kilin.project import (
    configure_project,
    prepare_project_workspace,
    synchronize_project_workspace,
)

staging_root = benchmarks_dir / "sandbox" / "_staging"
combined_source = staging_root / "kilin_and_tutorials"
if combined_source.exists():
    shutil.rmtree(combined_source)
combined_source.mkdir(parents=True)

_ignore = shutil.ignore_patterns("__pycache__", "*.pyc", ".DS_Store")
shutil.copytree(root / "kilin", combined_source / "kilin", ignore=_ignore)
shutil.copytree(
    root / "tutorials", combined_source / "tutorials", ignore=_ignore
)

workspace_root = benchmarks_dir / "sandbox"
project = prepare_project_workspace(
    source_directory=combined_source,
    type_id="code",
    workspace_root=workspace_root,
)

# A new source path can mean a brand-new workspace with default model settings.
# Record all indexed source types; packaged JSON describes defaults and
# project behavior and is essential for the current gold questions.
from kilin.model_defaults import (
    DEFAULT_CHAT_MODEL_MLX,
    DEFAULT_CHAT_MODEL_TORCH,
    DEFAULT_EMBEDDING_MODEL_MLX,
    DEFAULT_EMBEDDING_MODEL_TORCH,
)

DESIRED_MODEL_SETTINGS = {
    "chat_mlx": DEFAULT_CHAT_MODEL_MLX,
    "chat_torch": DEFAULT_CHAT_MODEL_TORCH,
    "embedding_mlx": DEFAULT_EMBEDDING_MODEL_MLX,
    "embedding_torch": DEFAULT_EMBEDDING_MODEL_TORCH,
    "extensions": ".py, .ipynb, .m, .md, .json",
}
if any(
    getattr(project.profile, key) != value
    for key, value in DESIRED_MODEL_SETTINGS.items()
    if key != "extensions"
) or ".json" not in project.profile.extensions:
    project = configure_project(project.root, DESIRED_MODEL_SETTINGS)
    # Newly indexed JSON files need an explicit snapshot update.
    project = synchronize_project_workspace(
        combined_source, project.root
    )
    print("Benchmark profile and source snapshot updated.")

print(f"Combined source staged at: {combined_source.relative_to(root)}")
print(f"Project root: {project.root.relative_to(root)}")
print(f"Project source: {project.source}")
print(f"Chat model (mlx): {project.profile.chat_mlx}")
print(f"Embedding model (mlx): {project.profile.embedding_mlx}")
print(
    f"Chunking: target={project.profile.target_tokens} "
    f"max={project.profile.max_tokens} "
    f"overlap={project.profile.overlap_tokens}"
)

## 4. Open a real RAGSession

This single call is loading the
embedding model, loading the chat model, and building (or reusing) the semantic, lexical, and code-graph
indexes. `progress_callback` prints KíLín's own structured progress events so a first run (with no
cached index yet) doesn't sit silently.


In [ ]:
from kilin.rag import RAGSession


def _print_progress(event):
    if event.stage in {"ready"} or event.current is None:
        print(f"[{event.percent:5.1f}%] {event.message}")


rag_session = RAGSession(project.root, progress_callback=_print_progress)

print(f"Backend: {rag_session.backend}")
print(f"Index: {rag_session.index_status}")
print(f"Chunks: {len(rag_session.chunks)}")

## 5. Configure answer guidance

Set a fixed optional answer instruction without modifying private chat
state. KíLín composes its effective system prompt from project-type
configuration on each turn; assigning to `_system_prompt` would not
provide a reliable benchmark override. Disabling the router keeps the
experiment focused on evidence retrieval, citations, and generation.


In [ ]:
USE_STRONGER_GUIDANCE = True
USE_ROUTER = False

# Keep the fixed evidence-search intent for citation benchmarks. A
# routed experiment can be run by setting USE_ROUTER to True, but
# tool-generated answers then measure a different retrieval pipeline.
_STRONGER_ANSWER_GUIDANCE = (
    "Answer only from the supplied project evidence. Cite each "
    "supported claim using [1], [2], and so on. If the evidence "
    "clearly does not answer the question, respond with exactly: "
    "Insufficient evidence. Do not guess or answer from general "
    "knowledge."
)

ANSWER_GUIDANCE = (
    _STRONGER_ANSWER_GUIDANCE if USE_STRONGER_GUIDANCE else None
)
print(f"Router enabled: {USE_ROUTER}")
print(f"Additional answer guidance: {ANSWER_GUIDANCE!r}")

## 6. Check gold citations against the real, chunked corpus

Confirm each question's gold citation(s) actually land inside a real chunk.


In [ ]:
from benchmarks.common import gold_evidence_indices


class _ChunkAsEvidence:
    """Adapt a real Chunk to the same (path, start_line, end_line, cell,
    citation) shape gold_evidence_indices expects from an Evidence --
    citation is unused for this check, only path/cell/line overlap matters.
    cell matters for a notebook-derived chunk: every cell's line numbers
    start at 1, so a gold citation naming a specific cell ("path#cell:1-8")
    needs the real cell number to disambiguate, not just the line range.
    """

    __slots__ = ("citation", "path", "start_line", "end_line", "cell")

    def __init__(self, chunk):
        self.citation = 0
        self.path = chunk.path
        self.start_line = chunk.start_line
        self.end_line = chunk.end_line
        self.cell = chunk.cell


chunk_evidence = [_ChunkAsEvidence(chunk) for chunk in rag_session.chunks]

unmatched = []
for task in tasks:
    if task.requires_abstention:
        continue
    if not gold_evidence_indices(chunk_evidence, task.gold_citations):
        unmatched.append(task.task_id)

if unmatched:
    print(
        f"WARNING: {len(unmatched)} question(s) have a gold citation "
        f"that matches no real chunk: {unmatched}"
    )
    print(
        "Check these against the current KíLín source before trusting their "
        "scores below."
    )
else:
    print(
        f"All {len(tasks) - sum(1 for t in tasks if t.requires_abstention)} "
        "answerable questions' gold citations match a real chunk."
    )

## 7. Ask every question for real

`rag_session.reset()` before each question keeps every turn an independent, single-turn conversation.
`rag_session.turns[-1].evidence` after `ask()` is the exact, real
`Evidence` KíLín retrieved and supplied for that answer.


In [ ]:
from tqdm.auto import tqdm

run_results: list[dict[str, object]] = []

for task in tqdm(tasks, desc="Asking", unit="question"):
    rag_session.reset()
    answer = rag_session.ask(
        task.question,
        use_router=USE_ROUTER,
        answer_guidance=ANSWER_GUIDANCE,
    )
    turn = rag_session.turns[-1]
    run_results.append(
        {
            "task_id": task.task_id,
            "question": task.question,
            "answer": answer,
            "evidence": [
                {
                    "citation": item.citation,
                    "path": str(item.path),
                    "start_line": item.start_line,
                    "end_line": item.end_line,
                    "symbol": item.symbol,
                    "text": item.text,
                    "cell": item.cell,
                }
                for item in turn.evidence
            ],
        }
    )

print(f"Asked {len(run_results)} questions.")

## 8. Score Tier 1 (mechanical)

Citation validity and coverage need the real `Evidence` objects each answer was actually given, so
these run against `run_results`, not against the gold set alone.


In [ ]:
from dataclasses import dataclass as _dataclass
from pathlib import Path as _Path

from benchmarks.common import (
    abstains,
    citation_coverage,
    citation_validity,
    verbatim_overlap_chars,
)

# Long verbatim overlaps need human review, except the prompt quote (A3).
VERBATIM_CHAR_THRESHOLD = 60


@_dataclass
class _EvidenceView:
    citation: int
    path: _Path
    start_line: int
    end_line: int
    symbol: object
    text: str
    cell: int | None = None


tasks_by_id = {task.task_id: task for task in tasks}
tier1_rows: list[dict[str, object]] = []

for row in run_results:
    task = tasks_by_id[row["task_id"]]
    evidence = [_EvidenceView(**item) for item in row["evidence"]]

    valid, invalid_indices = citation_validity(
        row["answer"], evidence_count=len(evidence)
    )
    coverage = (
        None
        if task.requires_abstention
        else citation_coverage(row["answer"], evidence, task.gold_citations)
    )
    abstained = abstains(row["answer"])
    verbatim_chars = verbatim_overlap_chars(row["answer"], evidence)

    tier1_rows.append(
        {
            "task_id": task.task_id,
            "requires_abstention": task.requires_abstention,
            "citation_valid": valid,
            "invalid_citation_indices": invalid_indices,
            "citation_coverage": coverage,
            "abstained": abstained,
            "abstention_correct": (
                (abstained if task.requires_abstention else not abstained)
            ),
            "verbatim_overlap_chars": verbatim_chars,
            "verbatim_flag": (
                verbatim_chars is not None
                and verbatim_chars >= VERBATIM_CHAR_THRESHOLD
            ),
        }
    )

valid_count = sum(1 for r in tier1_rows if r["citation_valid"])
abstention_correct_count = sum(
    1 for r in tier1_rows if r["abstention_correct"]
)
coverage_values = [
    r["citation_coverage"]
    for r in tier1_rows
    if r["citation_coverage"] is not None
]
verbatim_flags = [r["task_id"] for r in tier1_rows if r["verbatim_flag"]]

print(
    f"Citation validity: {valid_count}/{len(tier1_rows)} answers cite only "
    "real evidence indices."
)
print(
    f"Abstention correctness: {abstention_correct_count}/{len(tier1_rows)} "
    f"({abstention_count} deliberate negatives)."
)
if coverage_values:
    print(
        f"Mean citation coverage (answerable questions with retrieved gold "
        f"evidence, n={len(coverage_values)}): "
        f"{sum(coverage_values) / len(coverage_values):.2f}"
    )
else:
    print(
        "No answerable question had its gold evidence retrieved "
        "-- coverage undefined."
    )
print(
    f"Verbatim-copy flags (>= {VERBATIM_CHAR_THRESHOLD} chars, "
    f"A3 allows quotation): {verbatim_flags or 'none'}"
)

## 9. Export the Tier 2 grading worksheet

Faithfulness (does each cited claim actually hold up against the evidence it cites) and correctness
(does the answer satisfy its fact checklist) need a human reading the answer, by design.
This just lays out everything needed to grade Tier 2 by hand, question by question.


In [ ]:
worksheet_lines = []
for row, tier1 in zip(run_results, tier1_rows):
    task = tasks_by_id[row["task_id"]]
    worksheet_lines.append(
        f"## {task.task_id}"
        f"{' [abstention]' if task.requires_abstention else ''}"
    )
    worksheet_lines.append(f"**Question:** {task.question}")
    worksheet_lines.append("")
    worksheet_lines.append(f"**Answer:**\n{row['answer']}")
    worksheet_lines.append("")
    worksheet_lines.append("**Evidence supplied:**")
    for item in row["evidence"]:
        symbol = f" ({item['symbol']})" if item["symbol"] else ""
        worksheet_lines.append(
            f"- [{item['citation']}] {item['path']}:"
            f"{item['start_line']}-{item['end_line']}{symbol}"
        )
    worksheet_lines.append("")
    worksheet_lines.append("**Fact checklist:**")
    for fact in task.fact_checklist:
        worksheet_lines.append(f"- [ ] {fact}")
    worksheet_lines.append("")
    if task.reference_answer:
        worksheet_lines.append(
            "**Reference reply (from the fact checklist -- a comparison "
            "aid, not ground truth on its own):**"
        )
        worksheet_lines.append(task.reference_answer)
        worksheet_lines.append("")
    worksheet_lines.append(
        f"**Tier 1 (automatic):** citation_valid={tier1['citation_valid']}, "
        f"citation_coverage={tier1['citation_coverage']}, "
        f"abstained={tier1['abstained']}, "
        f"abstention_correct={tier1['abstention_correct']}, "
        f"verbatim_overlap_chars={tier1['verbatim_overlap_chars']}"
    )
    worksheet_lines.append("")
    worksheet_lines.append(
        "**Tier 2 (your read):** faithful? [ ]  checklist satisfied? [ ]  "
        "notes: "
    )
    worksheet_lines.append("\n---\n")

worksheet_text = "\n".join(worksheet_lines)
worksheet_path = benchmarks_dir / "results"
worksheet_path.mkdir(parents=True, exist_ok=True)

## 10. Save results

Tier 1 scores and the raw per-question data go into `benchmarks/results/*.json`. The Tier 2 worksheet is saved alongside it as its own markdown file,
timestamped the same way, so a graded copy can be diffed against the run it was graded from.


In [ ]:
import json as _json
from benchmarks.common import local_run_timestamp

results_dir = benchmarks_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)

timestamp, stamp = local_run_timestamp()

run_record = {
    "benchmark": (
        "KíLín dogfooding -- own repo, hand-authored gold questions "
        "(Area A: code)"
    ),
    "timestamp": timestamp,
    "chat_model": project.profile.chat_model(rag_session.backend),
    "embedding_model": project.profile.embedding_model(rag_session.backend),
    "backend": rag_session.backend,
    "index_status": rag_session.index_status,
    "chunk_count": len(rag_session.chunks),
    "question_count": len(tasks),
    "abstention_question_count": sum(
        1 for t in tasks if t.requires_abstention
    ),
    "gold_citation_unmatched": unmatched,
    "answer_guidance": ANSWER_GUIDANCE,
    "use_router": USE_ROUTER,
    "tier1_summary": {
        "citation_valid_count": valid_count,
        "abstention_correct_count": abstention_correct_count,
        "mean_citation_coverage": (
            sum(coverage_values) / len(coverage_values)
            if coverage_values
            else None
        ),
        "verbatim_flagged_task_ids": verbatim_flags,
    },
    "per_question": [
        {**row, "tier1": tier1} for row, tier1 in zip(run_results, tier1_rows)
    ],
}

output_path = results_dir / f"{stamp}_dogfood_kilin_code.json"
output_path.write_text(_json.dumps(run_record, indent=2))

worksheet_output_path = (
    results_dir / f"{stamp}_dogfood_kilin_code_tier2_worksheet.md"
)
worksheet_output_path.write_text(worksheet_text)

print(f"Saved: {output_path.relative_to(root)}")
print(f"Saved: {worksheet_output_path.relative_to(root)}")

## 11. Release the models


In [ ]:
rag_session.close()

print("Chat and embedding models released.")